In [ ]:
from pathlib import Path
import SimpleITK as sitk 
import numpy as np

def relabel_incorrect_necrosis(
    mask: sitk.Image,
    flair_scan: sitk.Image,
    csf_segmentation: sitk.Image,
    volume_threshold_total: int = 50,
    volume_threshold_pct: float = 15.0,
) -> sitk.Image:
    """
    Relabel necrosis in tumor segmentation to 3 labels (edema/NET, NCR & ET).
    - Closing of enhancement seals rims
    - Cavities and voxels gained by closing are protected as necrosis
    - Remaining necrosis undergoes intensity + volume filtering
    - Finally, hole filling and restoration of protected voxels
    """
    ### First check: volumes
    # Check volume of enhancement/necrosis, relabel all to 1 if very low
    mask_array = sitk.GetArrayFromImage(mask)
    new_mask = mask_array.copy()
    volume_enhancing = np.sum(mask_array == 3)
    volume_necrosis = np.sum(mask_array == 2)
    if volume_enhancing <= volume_threshold_total:
        mask_array[mask_array > 0] = 1
        out = sitk.GetImageFromArray(mask_array)
        out.CopyInformation(mask)
        return out
    elif volume_necrosis <= volume_threshold_total:
        mask_array[mask_array == 2] = 1
        out = sitk.GetImageFromArray(mask_array)
        out.CopyInformation(mask)
        return out

    ### Second check: encircled by enhancement -> find true necrosis
    # Closing of enhancement, in case of gaps
    enhancing_img = sitk.BinaryThreshold(mask, 3, 3, 1, 0)
    closed_enhancing_img = sitk.BinaryMorphologicalClosing(enhancing_img, 3 * [10])

    enhancing_array = sitk.GetArrayFromImage(enhancing_img).astype(bool)
    closed_enhancing_array = sitk.GetArrayFromImage(closed_enhancing_img).astype(bool)
    closing_difference = (
        closed_enhancing_array & ~enhancing_array
    )  # voxels gained by closing
    closing_difference &= mask_array == 2

    # Get cavities after hole filling
    filled_enhancing_img = sitk.BinaryFillhole(
        closed_enhancing_img, fullyConnected=True
    )
    enhancing_cavities_img = sitk.Subtract(filled_enhancing_img, closed_enhancing_img)
    cavities_array = sitk.GetArrayFromImage(enhancing_cavities_img).astype(bool)

    # Save cavities + closing_difference for now
    new_mask[cavities_array | closing_difference] = 10

    ### Third check: FLAIR intensities -> find necrotic cysts
    # Get intensity references edema / CSF in FLAIR
    flair_array = sitk.GetArrayFromImage(flair_scan)
    csf_array = sitk.GetArrayFromImage(csf_segmentation)
    csf_vals = flair_array[csf_array > 0]
    edema_vals = flair_array[mask_array == 1]
    if len(csf_vals) == 0 or len(edema_vals) == 0:
        print("Warning: empty CSF or edema reference, skipping necrosis relabeling")
        return mask
    csf_qhi = np.percentile(csf_vals, 95)
    edema_qlo = np.percentile(edema_vals, 5)

    if np.sum(mask_array == 2) == 0 or np.sum(mask_array == 3) == 0:
        print("No necrosis and/or enhancement present, skipping relabeling")
        return mask

    necrosis_array = new_mask == 2
    zs, ys, xs = np.where(necrosis_array)
    for z, y, x in zip(zs, ys, xs):
        val = flair_array[z, y, x]
        if not (csf_qhi <= val <= edema_qlo):
            new_mask[z, y, x] = 1  # relabel to edema

    # connected components volume filter (necrotic cyst is large)
    tumor_volume = np.sum(mask_array > 0)
    necrosis_img = sitk.GetImageFromArray((new_mask == 2).astype(np.uint8))
    necrosis_img.CopyInformation(mask)
    cc = sitk.ConnectedComponent(necrosis_img)
    stats = sitk.LabelShapeStatisticsImageFilter()
    stats.Execute(cc)
    for lbl in stats.GetLabels():
        blob = sitk.BinaryThreshold(cc, lbl, lbl, 1, 0)
        blob_arr = sitk.GetArrayFromImage(blob).astype(bool)
        blob_vol = stats.GetNumberOfPixels(lbl)
        if 100.0 * blob_vol / tumor_volume < volume_threshold_pct:
            new_mask[blob_arr & (new_mask == 2)] = 1  # too small → edema
        else:
            new_mask[blob_arr & (new_mask == 2)] = 2

    # hole filling of remaining component(s)
    necrosis_img = sitk.GetImageFromArray((new_mask == 2).astype(np.uint8))
    necrosis_img.CopyInformation(mask)
    filled = sitk.BinaryFillhole(necrosis_img, fullyConnected=True)
    filled_arr = sitk.GetArrayFromImage(filled).astype(bool)
    new_mask[filled_arr] = 2

    # restore protected necrosis
    new_mask[new_mask == 10] = 2

    # preserve original enhancement (don’t overwrite necrosis)
    new_mask[(enhancing_array) & (new_mask != 2)] = 3

    out = sitk.GetImageFromArray(new_mask.astype(np.uint8))
    out.CopyInformation(mask)
    return out